In [ ]:
import wikipediaapi

In [ ]:
wiki = wikipediaapi.Wikipedia(user_agent='Bing-Chilling', language='en')

mh3 = wiki.page("Monster_Hunter_Tri")

In [ ]:
print(f"Page title: {mh3.title}")
print(f"Page Summary: {mh3.summary}")

In [ ]:
mh3.text

In [ ]:
def print_sections(sections, level=0):
    for s in sections:
        print("%s: %s - %s" % ("*" * (level + 1), s.title, s.text[0:40]))
        print_sections(s.sections, level + 1)


print_sections(mh3.sections)

In [ ]:
import os

path = r"C:\Users\Dhyey\Documents\Python\Kaggle\Datasets\RAG\documents"
page_list = [os.path.splitext(filename)[0] for filename in os.listdir(path)]
page_list.pop(0)

for i, page in enumerate(page_list):
    page_list[i] = page.replace(r"__",": ")
    # print(page)

if page_list[-1] == "wikipedia_pages":
    page_list.pop(-1) 
print(page_list)

In [ ]:
pd = wiki.pages(page_list)
pd

for page in pd.keys():
    print(f"{page}: {pd[page].text[:50]}")

In [ ]:
wiki_dict = {}
for key in pd.keys():
    wiki_dict[key] = pd[key].text

In [ ]:
STOP = ["\n== References", "\n== External links", "\n== See also", "\n== Notes"]

def strip_tail(text):
    cuts = [text.find(s) for s in STOP if text.find(s) != -1]
    return text[:min(cuts)] if cuts else text

for key in wiki_dict.keys():
    wiki_dict[key] = strip_tail(wiki_dict[key])

In [ ]:
wiki_dict["List_of_best-selling_video_games"]

In [ ]:
import json
from pathlib import Path

OUT = Path("../documents/wikipedia_pages.json")
OUT.parent.mkdir(parents=True, exist_ok=True)

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(wiki_dict, f, ensure_ascii=False, indent=2)

In [ ]:
# rag/ingest.py
import json
from pathlib import Path

from langchain_core.documents import Document

PAGES_FILE = Path(r"C:\Users\Dhyey\Documents\Python\Kaggle\Datasets\RAG\documents\wikipedia_pages.json")

def load_documents(pages_file: Path = PAGES_FILE) -> list[Document]:
    with open(pages_file, "r", encoding="utf-8") as f:
        pages = json.load(f)

    return [
        Document(page_content=text, metadata={"source": title})
        for title, text in pages.items()
    ]

documents = load_documents(PAGES_FILE)

In [ ]:
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

from langchain_text_splitters import RecursiveCharacterTextSplitter

def get_splitter():
    return RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n== ", "\n=== ", "\n\n", "\n", ". ", " ", ""],
    )

In [ ]:
splitter = get_splitter()
splitter.split_documents(documents)

In [ ]:
def print_categorymembers(categorymembers, level=0, max_level=1):
    for c in categorymembers.values():
        print("%s: %s (ns: %d)" % ("*" * (level + 1), c.title, c.ns))
        if c.ns == wikipediaapi.Namespace.CATEGORY and level < max_level:
            print_categorymembers(c.categorymembers, level=level + 1, max_level=max_level)


import wikipediaapi
wiki = wikipediaapi.Wikipedia(user_agent='Bing-Chilling', language='en')

cat = wiki.page("Category:Monster_Hunter")
print("Category members: Category:Monster_Hunter")
print_categorymembers(cat.categorymembers)

In [ ]:
import json
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

API = "https://monsterhunter.fandom.com/api.php"
OUT = Path("../documents/mh_monsters.json")
HEADERS = {"User-Agent": "MH-RAG/0.1 (portfolio project)"}
MIN_PAGE_CHARS = 400

# Block-level tags. Text is joined within a block, split between blocks,
# so inline links no longer fragment sentences.
BLOCK_TAGS = ["p", "h1", "h2", "h3", "h4", "h5", "li", "dd", "dt", "blockquote"]

DROP_SELECTORS = [
    "table",
    "aside.portable-infobox",
    "div.navbox",
    "div.toc",
    "sup.reference",
    "span.mw-editsection",
    "figure",
    "figcaption",
    "div.thumb",
    "div.mw-references-wrap",
    "div.notice",
    "div.gallery",
]

# Non-article pages that slip through the category scan
SKIP_TITLE_PATTERNS = ["(file)", "/Gallery", "/Videos", "(disambiguation)"]


def tidy(text):
    """Collapse whitespace and repair spacing around punctuation."""
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([.,;:!?)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    return text


def infobox_text(soup):
    """Pull the infobox out as label-value prose before it gets stripped."""
    box = soup.select_one("aside.portable-infobox")
    if not box:
        return ""

    parts = []
    for title in box.select("h2.pi-title[data-source]"):
        value = title.get_text(" ", strip=True)
        if value:
            parts.append(f"{title.get('data-source')}: {tidy(value)}")

    for item in box.select("div.pi-data"):
        label = item.select_one(".pi-data-label")
        value = item.select_one(".pi-data-value")
        if label and value:
            label_text = tidy(label.get_text(" ", strip=True))
            value_text = tidy(value.get_text(" ", strip=True))
            if label_text and value_text:
                parts.append(f"{label_text}: {value_text}")

    return "\n".join(parts)


def body_text(soup):
    """Extract prose block by block so inline links stay inside sentences."""
    blocks = []
    for el in soup.find_all(BLOCK_TAGS):
        text = tidy(el.get_text(" ", strip=True))
        if text:
            blocks.append(text)
    return "\n".join(blocks)


def fetch_page_text(title):
    r = requests.get(API, params={
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "formatversion": 2,
    }, headers=HEADERS, timeout=30)

    if r.status_code != 200:
        return None
    data = r.json()
    if "parse" not in data:
        return None

    soup = BeautifulSoup(data["parse"]["text"], "html.parser")

    facts = infobox_text(soup)          # extract before decomposing
    for selector in DROP_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()
    body = body_text(soup)

    return f"{facts}\n\n{body}".strip() if facts else body


def fetch_all(titles):
    pages, failed = {}, []
    todo = [t for t in titles if not any(p in t for p in SKIP_TITLE_PATTERNS)]
    skipped = len(titles) - len(todo)
    if skipped:
        print(f"Skipping {skipped} non-article titles.")

    bar = tqdm(todo, unit="page", desc="fetching")
    for title in bar:
        try:
            text = fetch_page_text(title)
        except Exception as e:
            failed.append((title, type(e).__name__))
            continue

        if text and len(text) >= MIN_PAGE_CHARS:
            pages[title] = text
        else:
            failed.append((title, "empty or too short"))

        chars = sum(len(v) for v in pages.values())
        bar.set_postfix_str(f"{title[:24]} | {len(pages)} kept | {chars:,} chars")
        time.sleep(0.3)

    bar.close()
    if failed:
        print(f"\n{len(failed)} failed or skipped, e.g. {failed[:5]}")
    return pages


sample = fetch_all(titles[:5])
for t, x in sample.items():
    print(f"\n--- {t} ({len(x)} chars) ---")
    print(x[:600])

In [ ]:
pages = fetch_all(titles)

total = sum(len(v) for v in pages.values())
print(f"\nFetched {len(pages)} pages, {total:,} characters.")
print(f"Mean length: {total // max(len(pages), 1):,} chars")

OUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)
print(f"Saved to {OUT}")

In [ ]:
import time

def category_members(category, namespace=0):
    """All article titles in a category."""
    titles = []
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": category,
        "cmlimit": "max",
        "cmnamespace": namespace,
        "format": "json",
    }
    while True:
        r = requests.get(API, params=params, headers=HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        titles.extend(m["title"] for m in data["query"]["categorymembers"])
        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.2)
    return titles


large = category_members("Category:Large Monsters")
small = category_members("Category:Small Monsters")
titles = sorted(set(large) | set(small))

print(f"Large: {len(large)}, Small: {len(small)}, unique total: {len(titles)}")
print(titles[:10])

In [ ]:
import json
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

API = "https://monsterhunterwiki.org/api.php"
OUT = Path("../documents/mh_wiki_monsters.json")
HEADERS = {"User-Agent": "MH-RAG/0.1 (portfolio project)"}

MIN_PAGE_CHARS = 400
MIN_BODY_CHARS = 300          # prose alone, excluding the infobox

INFOBOX_SELECTOR = "table.monster-game-info"

# This wiki uses <h1> for section headings, not <h2>
HEADING_TAGS = {"h1", "h2", "h3", "h4", "h5", "h6"}
BLOCK_TAGS = ["p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "dd", "dt", "blockquote"]

DROP_SELECTORS = [
    "table",
    "div.navbarheader",
    "div.navbar-green",
    "div.toc",
    "sup.reference",
    "span.mw-editsection",
    "figure",
    "figcaption",
    "div.thumb",
    "div.gallery",
    "div.mw-references-wrap",
    ".mw-ext-cite-error",      # "Cite error: <ref> tags exist for..."
    "span.error",
]

SKIP_NS = ("File:", "Help:", "Category:", "Template:", "Special:", "Talk:")

# Pages whose only content is the legend explaining the stat tables we strip.
# Their monster data lived in those tables, so nothing usable remains.
BOILERPLATE_MARKERS = (
    "The following information is for",
    "Part - the name of the part",
    "the effectiveness of Cutting, Blunt, and Shot",
)


def tidy(text):
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\s+([.,;:!?)\]])", r"\1", text)
    text = re.sub(r"([(\[])\s+", r"\1", text)
    text = re.sub(r",\s*\)", ")", text)          # "(Dragon,)" -> "(Dragon)"
    text = re.sub(r"\(\s*\)", "", text)          # parens left empty by icons
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip(" ,;")


def table_to_text(table):
    """Turn a wikitable into label-value lines. Handles both vertical
    (th | td per row) and horizontal (header row then data row) layouts."""
    parts = []
    pending_headers = None
    for row in table.find_all("tr"):
        headers = row.find_all("th", recursive=False)
        cells = row.find_all("td", recursive=False)
        if headers and cells:
            for h, c in zip(headers, cells):
                label = tidy(h.get_text(" ", strip=True))
                value = tidy(c.get_text(" ", strip=True))
                if label and value:
                    parts.append(f"{label}: {value}")
            pending_headers = None
        elif headers and not cells:
            pending_headers = [tidy(h.get_text(" ", strip=True)) for h in headers]
        elif cells and pending_headers:
            values = [tidy(c.get_text(" ", strip=True)) for c in cells]
            for label, value in zip(pending_headers, values):
                if label and value:
                    parts.append(f"{label}: {value}")
            pending_headers = None
    return "\n".join(parts)


def infobox_text(soup):
    table = soup.select_one(INFOBOX_SELECTOR)
    if not table:
        return ""

    for sup in table.find_all("sup"):
        sup.decompose()

    # element and status values are icons; the name lives on the parent link
    for img in table.find_all("img"):
        a = img.find_parent("a")
        href = a.get("href", "") if a else ""
        if not a or any(href.startswith(f"/wiki/{ns}") for ns in SKIP_NS):
            img.replace_with("")                 # renders, help links, etc.
            continue
        label = a.get("title") or href.rsplit("/", 1)[-1].replace("_", " ")
        label = re.sub(r"\s+Element$", "", label)
        img.replace_with(f" {label}, " if label else "")

    return table_to_text(table)


def body_text(soup):
    """Emit prose blocks, keeping a heading only when real content follows it.

    Sections whose content was in stripped tables or galleries otherwise leave
    long runs of consecutive headings with nothing beneath them.
    """
    blocks = []
    pending = []
    for el in soup.find_all(BLOCK_TAGS):
        text = tidy(el.get_text(" ", strip=True))
        if not text:
            continue
        if el.name in HEADING_TAGS:
            pending = [text]          # nearest heading only; earlier ones were unbacked
            continue
        if pending:
            blocks.extend(pending)
            pending = []
        blocks.append(text)
    return "\n".join(blocks)


def fetch_page_text(title):
    r = requests.get(API, params={
        "action": "parse",
        "page": title,
        "prop": "text",
        "format": "json",
        "formatversion": 2,
    }, headers=HEADERS, timeout=30)

    if r.status_code != 200:
        return None
    data = r.json()
    if "parse" not in data:
        return None

    soup = BeautifulSoup(data["parse"]["text"], "html.parser")

    facts = infobox_text(soup)          # extract before decomposing
    for selector in DROP_SELECTORS:
        for tag in soup.select(selector):
            tag.decompose()
    body = body_text(soup)

    # Judge the stub check on prose alone: a page with a populated infobox but
    # no descriptive text is still a stub, and checking the combined length
    # would let it through on the strength of the infobox.
    if any(m in body for m in BOILERPLATE_MARKERS):
        return None
    if len(body) < MIN_BODY_CHARS:
        return None

    return "\n\n".join(p for p in [facts, body] if p).strip()


def fetch_all(titles):
    pages, failed = {}, []
    bar = tqdm(titles, unit="page", desc="fetching")
    for title in bar:
        try:
            text = fetch_page_text(title)
        except Exception as e:
            failed.append((title, type(e).__name__))
            continue

        if text and len(text) >= MIN_PAGE_CHARS:
            pages[title] = text
        else:
            failed.append((title, "stub, empty, or too short"))

        chars = sum(len(v) for v in pages.values())
        bar.set_postfix_str(f"{title[:24]} | {len(pages)} kept | {chars:,} chars")
        time.sleep(0.3)

    bar.close()
    if failed:
        print(f"\n{len(failed)} failed or skipped, e.g. {failed[:5]}")
    return pages


sample = fetch_all(titles[:5])
for t, x in sample.items():
    print(f"\n--- {t} ({len(x)} chars) ---")
    print(x[:700])

In [ ]:
pages = fetch_all(titles)

total = sum(len(v) for v in pages.values())
print(f"\nFetched {len(pages)} of {len(titles)} pages, {total:,} characters.")
print(f"Mean length: {total // max(len(pages), 1):,} chars")

OUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)
print(f"Saved to {OUT}")

In [ ]:
import json
from pathlib import Path

with open(Path("../documents/mh_wiki_monsters.json"), encoding="utf-8") as f:
    mh = json.load(f)

missing = [t for t, x in mh.items() if "Classification:" not in x]
print(f"{len(missing)} of {len(mh)} pages lack an infobox")
print(missing[:15])